In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:

import os

for root, dirs, files in os.walk("/kaggle/input"):
    level = root.replace("/kaggle/input", "").count(os.sep)

    if level <= 3:
        indent = "    " * level
        print(f"{indent}{os.path.basename(root)}/")

        if level < 3:
            for file in files[:5]:
                print(f"{indent}    {file}")


In [ ]:

from pathlib import Path

DATA_ROOT = Path("/kaggle/input/datasets/gti-upm/leapgestrecog")

print("Dataset root exists:", DATA_ROOT.exists())
print("\nTop-level contents:")

for item in sorted(DATA_ROOT.iterdir()):
    print(" ", item.name, " [folder]" if item.is_dir() else " [file]")

print("\nFolder structure (first 3 levels):")

for path in sorted(DATA_ROOT.rglob("*")):
    relative = path.relative_to(DATA_ROOT)

    if len(relative.parts) <= 3:
        if path.is_dir():
            print("  " * (len(relative.parts) - 1) + relative.name + "/")
        elif path.suffix.lower() == ".png":
            print("  " * (len(relative.parts) - 1) + relative.name)


In [ ]:

from pathlib import Path
import pandas as pd

DATA_ROOT = Path(
    "/kaggle/input/datasets/gti-upm/leapgestrecog/leapGestRecog"
)

assert DATA_ROOT.exists(), f"Directory not found: {DATA_ROOT}"

subjects = sorted(
    p for p in DATA_ROOT.iterdir()
    if p.is_dir() and p.name.isdigit()
)

print("Number of subjects:", len(subjects))
print("Subject IDs:", [p.name for p in subjects])

records = []

for subject_dir in subjects:
    for gesture_dir in sorted(subject_dir.iterdir()):
        if not gesture_dir.is_dir():
            continue

        for image_path in gesture_dir.iterdir():
            if image_path.suffix.lower() in {".png", ".jpg", ".jpeg"}:
                records.append({
                    "path": str(image_path),
                    "subject": subject_dir.name,
                    "gesture": gesture_dir.name
                })

df = pd.DataFrame(records)

print("\nTotal images:", len(df))
print("\nImages per subject:")
print(df.groupby("subject").size())

print("\nImages per gesture:")
print(df.groupby("gesture").size())

print("\nImages per subject and gesture:")
print(df.pivot_table(
    index="subject",
    columns="gesture",
    values="path",
    aggfunc="count",
    fill_value=0
))


In [ ]:

# Step 4: Subject-wise data splitting

TRAIN_SUBJECTS = [f"{i:02d}" for i in range(8)]
VAL_SUBJECT = "08"
TEST_SUBJECT = "09"

train_df = df[df["subject"].isin(TRAIN_SUBJECTS)].copy()
val_df = df[df["subject"] == VAL_SUBJECT].copy()
test_df = df[df["subject"] == TEST_SUBJECT].copy()

# Verify the split
train_subjects = set(train_df["subject"])
val_subjects = set(val_df["subject"])
test_subjects = set(test_df["subject"])

assert train_subjects.isdisjoint(val_subjects)
assert train_subjects.isdisjoint(test_subjects)
assert val_subjects.isdisjoint(test_subjects)

assert len(train_df) == 16000
assert len(val_df) == 2000
assert len(test_df) == 2000

assert set(train_df["gesture"]) == set(df["gesture"])
assert set(val_df["gesture"]) == set(df["gesture"])
assert set(test_df["gesture"]) == set(df["gesture"])

print("TRAINING")
print("Subjects:", sorted(train_subjects))
print("Images:", len(train_df))

print("\nVALIDATION")
print("Subjects:", sorted(val_subjects))
print("Images:", len(val_df))

print("\nFINAL TEST")
print("Subjects:", sorted(test_subjects))
print("Images:", len(test_df))

print("\nNo subject overlap:", True)


In [ ]:

import matplotlib.pyplot as plt
from PIL import Image

# One sample image for each gesture
sample_df = (
    train_df.sort_values("path")
    .groupby("gesture", sort=True)
    .first()
    .reset_index()
)

fig, axes = plt.subplots(2, 5, figsize=(15, 7))

for ax, (_, row) in zip(axes.flat, sample_df.iterrows()):
    image = Image.open(row["path"]).convert("L")

    ax.imshow(image, cmap="gray")
    ax.set_title(row["gesture"])
    ax.axis("off")

plt.suptitle("LeapGestRecog: One Sample per Gesture", fontsize=15)
plt.tight_layout()
plt.show()

# Verify image dimensions
print("Sample image dimensions:")
for _, row in sample_df.iterrows():
    with Image.open(row["path"]) as image:
        print(f"{row['gesture']}: {image.size}, mode={image.mode}")


In [ ]:

import tensorflow as tf
import matplotlib.pyplot as plt
from PIL import Image

# Select one verified training image
sample_path = train_df.iloc[0]["path"]

# Load original grayscale image
original = Image.open(sample_path).convert("L")

# Resize while preserving the original aspect ratio
resized = original.resize((256, 96), Image.Resampling.LANCZOS)

# Convert to normalized float32 values
normalized = tf.convert_to_tensor(resized, dtype=tf.float32) / 255.0

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].imshow(original, cmap="gray")
axes[0].set_title(f"Original: {original.size}")

axes[1].imshow(resized, cmap="gray")
axes[1].set_title(f"Resized: {resized.size}")

axes[2].imshow(normalized.numpy(), cmap="gray", vmin=0, vmax=1)
axes[2].set_title("Normalized: [0, 1]")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

print("Original size:", original.size)
print("Resized size:", resized.size)
print("Normalized shape:", normalized.shape)
print("Normalized range:",
      float(tf.reduce_min(normalized)),
      float(tf.reduce_max(normalized)))


In [ ]:

import numpy as np
import tensorflow as tf

IMG_WIDTH = 256
IMG_HEIGHT = 96
BATCH_SIZE = 32

# Assign a stable numerical label to each gesture
classes = sorted(df["gesture"].unique())
class_to_idx = {name: idx for idx, name in enumerate(classes)}

print("Gesture classes:")
print(class_to_idx)

def preprocess_image(path, label):
    image_bytes = tf.io.read_file(path)
    image = tf.io.decode_image(
        image_bytes,
        channels=1,
        expand_animations=False
    )

    image.set_shape([None, None, 1])

    image = tf.image.resize(
        image,
        [IMG_HEIGHT, IMG_WIDTH]
    )

    image = tf.cast(image, tf.float32) / 255.0

    return image, label

def make_dataset(frame, training=False):
    paths = frame["path"].to_numpy()
    labels = frame["gesture"].map(class_to_idx).to_numpy(
        dtype=np.int32
    )

    dataset = tf.data.Dataset.from_tensor_slices(
        (paths, labels)
    )

    if training:
        dataset = dataset.shuffle(
            buffer_size=len(frame),
            seed=42,
            reshuffle_each_iteration=True
        )

    dataset = dataset.map(
        preprocess_image,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset

train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df)
test_ds = make_dataset(test_df)

# Verify one batch
images, labels = next(iter(train_ds))

print("\nImage batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Image data type:", images.dtype)
print(
    "Pixel range:",
    float(tf.reduce_min(images)),
    float(tf.reduce_max(images))
)
print("Labels in first batch:", labels[:10].numpy())

assert images.shape[1:] == (96, 256, 1)
assert len(classes) == 10
assert float(tf.reduce_min(images)) >= 0.0
assert float(tf.reduce_max(images)) <= 1.0


In [ ]:

import tensorflow as tf
from tensorflow.keras import layers, models

tf.keras.backend.clear_session()

model = models.Sequential([
    layers.Input(shape=(96, 256, 1)),

    # Block 1
    layers.Conv2D(32, 3, padding="same", use_bias=False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D(pool_size=2),

    # Block 2
    layers.Conv2D(64, 3, padding="same", use_bias=False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D(pool_size=2),

    # Block 3
    layers.Conv2D(128, 3, padding="same", use_bias=False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D(pool_size=2),

    # Block 4
    layers.Conv2D(192, 3, padding="same", use_bias=False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D(pool_size=2),

    # Classification head
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.4),
    layers.Dense(10, activation="softmax")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

print("\nOutput shape:", model.output_shape)
print("Total parameters:", model.count_params())

assert model.input_shape == (None, 96, 256, 1)
assert model.output_shape == (None, 10)


In [ ]:

import tensorflow as tf
import os

OUTPUT_DIR = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=os.path.join(
            OUTPUT_DIR, "best_gesture_model.keras"
        ),
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

print("Training callbacks configured.")
print("Checkpoint:", os.path.join(
    OUTPUT_DIR, "best_gesture_model.keras"
))
print("Maximum epochs: 30")
print("Early stopping patience: 5")


In [ ]:

EPOCHS = 30

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

print("\nTraining completed.")
print("Epochs completed:", len(history.history["loss"]))
print(
    "Best validation accuracy:",
    f"{max(history.history['val_accuracy']) * 100:.2f}%"
)
print(
    "Lowest validation loss:",
    f"{min(history.history['val_loss']):.4f}"
)


In [ ]:

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Accuracy
axes[0].plot(
    history.history["accuracy"],
    marker="o",
    label="Training accuracy"
)
axes[0].plot(
    history.history["val_accuracy"],
    marker="o",
    label="Validation accuracy"
)
axes[0].set_title("Training vs Validation Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(
    history.history["loss"],
    marker="o",
    label="Training loss"
)
axes[1].plot(
    history.history["val_loss"],
    marker="o",
    label="Validation loss"
)
axes[1].set_title("Training vs Validation Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns

# Load the best checkpoint selected using validation subject 08
best_model = tf.keras.models.load_model(
    "/kaggle/working/best_gesture_model.keras"
)

# Predict on the completely unseen subject 09
y_true = np.concatenate([
    labels.numpy() for _, labels in test_ds
])

probabilities = best_model.predict(test_ds, verbose=1)
y_pred = np.argmax(probabilities, axis=1)

test_accuracy = accuracy_score(y_true, y_pred)
test_macro_f1 = f1_score(
    y_true, y_pred, average="macro"
)

print("UNSEEN SUBJECT EVALUATION")
print("Test subject:", TEST_SUBJECT)
print("Number of test images:", len(y_true))
print(f"Accuracy: {test_accuracy * 100:.2f}%")
print(f"Macro F1: {test_macro_f1:.4f}")

print("\nClassification report:")
print(classification_report(
    y_true,
    y_pred,
    labels=list(range(len(classes))),
    target_names=classes,
    zero_division=0
))

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=list(range(len(classes)))
)

plt.figure(figsize=(11, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=classes,
    yticklabels=classes
)
plt.xlabel("Predicted gesture")
plt.ylabel("Actual gesture")
plt.title("Confusion Matrix — Unseen Subject 09")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:

import hashlib
from collections import defaultdict

hash_to_subjects = defaultdict(set)
hash_to_paths = defaultdict(list)

for row in df[["path", "subject"]].itertuples(index=False):
    path, subject = row

    with open(path, "rb") as f:
        digest = hashlib.sha256(f.read()).hexdigest()

    hash_to_subjects[digest].add(subject)
    hash_to_paths[digest].append(path)

cross_subject_duplicates = {
    digest: paths
    for digest, paths in hash_to_paths.items()
    if len(hash_to_subjects[digest]) > 1
}

print("Total images checked:", len(df))
print("Unique image hashes:", len(hash_to_paths))
print(
    "Exact duplicate images across subjects:",
    sum(len(paths) - 1 for paths in cross_subject_duplicates.values())
)

if cross_subject_duplicates:
    print("\nExamples:")
    for paths in list(cross_subject_duplicates.values())[:5]:
        print(paths)
else:
    print("No exact duplicate image files found across subjects.")


In [ ]:

# Count exact duplicate hashes crossing split boundaries

split_by_subject = {
    **{s: "train" for s in TRAIN_SUBJECTS},
    VAL_SUBJECT: "validation",
    TEST_SUBJECT: "test"
}

cross_split_groups = []

for digest, paths in hash_to_paths.items():
    if len(paths) < 2:
        continue

    group = df[df["path"].isin(paths)]
    splits = set(
        group["subject"].map(split_by_subject)
    )

    if len(splits) > 1:
        cross_split_groups.append({
            "hash": digest,
            "splits": sorted(splits),
            "subjects": sorted(group["subject"].unique()),
            "paths": paths
        })

print("Duplicate hash groups crossing splits:",
      len(cross_split_groups))

for split_name in ["train", "validation", "test"]:
    count = sum(
        split_name in group["splits"]
        for group in cross_split_groups
    )
    print(f"Groups involving {split_name}: {count}")

print("\nExamples of cross-split duplicates:")
for group in cross_split_groups[:10]:
    print(
        "Splits:", group["splits"],
        "| Subjects:", group["subjects"],
        "| Paths:", group["paths"]
    )


In [ ]:

import os
import json
import pandas as pd
import numpy as np

# Verify that all required objects exist in this notebook session
required = [
    "df", "classes", "class_to_idx", "TRAIN_SUBJECTS",
    "VAL_SUBJECT", "TEST_SUBJECT", "train_df", "val_df",
    "test_df", "history", "test_accuracy", "test_macro_f1",
    "cross_split_groups", "best_model"
]

missing = [name for name in required if name not in globals()]

if missing:
    raise RuntimeError(
        f"Required variables are missing: {missing}. "
        "Run the relevant earlier cells before saving."
    )

OUTPUT_DIR = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save the trained model
model_path = os.path.join(
    OUTPUT_DIR, "leap_gesture_cnn.keras"
)
best_model.save(model_path)

# Save the class mapping
mapping_path = os.path.join(
    OUTPUT_DIR, "class_mapping.json"
)
with open(mapping_path, "w") as f:
    json.dump(class_to_idx, f, indent=2)

# Save measured experiment results
report = {
    "dataset": "LeapGestRecog",
    "total_images": int(len(df)),
    "number_of_classes": int(len(classes)),
    "image_dimensions": [96, 256, 1],
    "model": "Custom CNN",
    "total_parameters": int(best_model.count_params()),
    "training_subjects": list(TRAIN_SUBJECTS),
    "validation_subject": VAL_SUBJECT,
    "test_subject": TEST_SUBJECT,
    "training_images": int(len(train_df)),
    "validation_images": int(len(val_df)),
    "test_images": int(len(test_df)),
    "best_validation_accuracy": float(
        max(history.history["val_accuracy"])
    ),
    "unseen_subject_accuracy": float(test_accuracy),
    "unseen_subject_macro_f1": float(test_macro_f1),
    "duplicate_groups_crossing_splits": int(
        len(cross_split_groups)
    )
}

report_path = os.path.join(
    OUTPUT_DIR, "experiment_report.json"
)
with open(report_path, "w") as f:
    json.dump(report, f, indent=2)

# Verify the files exist and have nonzero size
for path in [model_path, mapping_path, report_path]:
    assert os.path.isfile(path) and os.path.getsize(path) > 0
    print(f"Verified: {path} ({os.path.getsize(path):,} bytes)")

print("\nReported unseen-subject accuracy:",
      f"{test_accuracy * 100:.2f}%")
print("Reported macro F1:", f"{test_macro_f1:.4f}")


In [ ]:

folds = []
subjects = sorted(df["subject"].unique())

for i, test_subject in enumerate(subjects):
    remaining = [s for s in subjects if s != test_subject]

    # Rotate validation subjects across folds
    val_subject = remaining[i % len(remaining)]

    fit_subjects = [
        s for s in remaining if s != val_subject
    ]

    fit_df_fold = df[
        df["subject"].isin(fit_subjects)
    ]
    val_df_fold = df[df["subject"] == val_subject]
    test_df_fold = df[df["subject"] == test_subject]

    assert len(fit_df_fold) == 16000
    assert len(val_df_fold) == 2000
    assert len(test_df_fold) == 2000

    assert set(fit_df_fold["subject"]).isdisjoint(
        val_df_fold["subject"]
    )
    assert set(fit_df_fold["subject"]).isdisjoint(
        test_df_fold["subject"]
    )
    assert set(val_df_fold["subject"]).isdisjoint(
        test_df_fold["subject"]
    )

    folds.append({
        "test_subject": test_subject,
        "validation_subject": val_subject,
        "fit_subjects": fit_subjects
    })

fold_summary = pd.DataFrame([
    {
        "fold": i + 1,
        "test_subject": f["test_subject"],
        "validation_subject": f["validation_subject"],
        "fit_subjects": ", ".join(f["fit_subjects"])
    }
    for i, f in enumerate(folds)
])

display(fold_summary)

print("\nValidation subject assignment counts:")
print(
    pd.Series([f["validation_subject"] for f in folds])
      .value_counts()
      .sort_index()
)

print("\nAll 10 folds passed subject-overlap checks.")


In [ ]:

import tensorflow as tf
import numpy as np
import pandas as pd

# Explicit configuration
IMG_HEIGHT = 96
IMG_WIDTH = 256
BATCH_SIZE = 32

# Reconstruct the class mapping from the verified class list
classes = sorted(df["gesture"].unique())
class_to_idx = {name: i for i, name in enumerate(classes)}

assert len(classes) == 10
assert len(folds) == 10

def build_cnn():
    """Return a fresh CNN with randomly initialized weights."""
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 1)),

        tf.keras.layers.Conv2D(
            32, 3, padding="same", use_bias=False
        ),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Activation("relu"),
        tf.keras.layers.MaxPooling2D(2),

        tf.keras.layers.Conv2D(
            64, 3, padding="same", use_bias=False
        ),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Activation("relu"),
        tf.keras.layers.MaxPooling2D(2),

        tf.keras.layers.Conv2D(
            128, 3, padding="same", use_bias=False
        ),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Activation("relu"),
        tf.keras.layers.MaxPooling2D(2),

        tf.keras.layers.Conv2D(
            192, 3, padding="same", use_bias=False
        ),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Activation("relu"),
        tf.keras.layers.MaxPooling2D(2),

        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.Dense(10, activation="softmax")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model


def make_fold_dataset(frame, training=False):
    """Create a fresh dataset using explicit paths and labels."""
    paths = frame["path"].to_numpy()
    labels = frame["gesture"].map(class_to_idx).to_numpy(
        dtype=np.int32
    )

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    if training:
        ds = ds.shuffle(
            buffer_size=len(frame),
            seed=42,
            reshuffle_each_iteration=True
        )

    def load_image(path, label):
        image = tf.io.read_file(path)
        image = tf.io.decode_png(image, channels=1)
        image = tf.image.resize(
            image, [IMG_HEIGHT, IMG_WIDTH]
        )
        image = tf.cast(image, tf.float32) / 255.0
        return image, label

    ds = ds.map(
        load_image,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


# Inspect fold 1 without training
fold = folds[0]

fit_df_fold = df[
    df["subject"].isin(fold["fit_subjects"])
].copy()

val_df_fold = df[
    df["subject"] == fold["validation_subject"]
].copy()

test_df_fold = df[
    df["subject"] == fold["test_subject"]
].copy()

assert set(fit_df_fold["subject"]).isdisjoint(
    val_df_fold["subject"]
)
assert set(fit_df_fold["subject"]).isdisjoint(
    test_df_fold["subject"]
)
assert set(val_df_fold["subject"]).isdisjoint(
    test_df_fold["subject"]
)

train_ds_fold = make_fold_dataset(
    fit_df_fold, training=True
)
val_ds_fold = make_fold_dataset(val_df_fold)
test_ds_fold = make_fold_dataset(test_df_fold)

images, labels = next(iter(train_ds_fold))
model_check = build_cnn()

print("Fold configuration:")
print("Fitting subjects:", fold["fit_subjects"])
print("Validation subject:", fold["validation_subject"])
print("Test subject:", fold["test_subject"])

print("\nImage counts:")
print("Fit:", len(fit_df_fold))
print("Validation:", len(val_df_fold))
print("Test:", len(test_df_fold))

print("\nTraining batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Model input shape:", model_check.input_shape)
print("Model output shape:", model_check.output_shape)
print("Model parameters:", model_check.count_params())

assert images.shape == (BATCH_SIZE, 96, 256, 1)
assert model_check.output_shape == (None, 10)

print("\nFold pipeline verification passed.")


In [ ]:

import tensorflow as tf
import numpy as np
import pandas as pd

# Explicit configuration
IMG_HEIGHT = 96
IMG_WIDTH = 256
BATCH_SIZE = 32

# Reconstruct the class mapping from the verified class list
classes = sorted(df["gesture"].unique())
class_to_idx = {name: i for i, name in enumerate(classes)}

assert len(classes) == 10
assert len(folds) == 10

def build_cnn():
    """Return a fresh CNN with randomly initialized weights."""
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 1)),

        tf.keras.layers.Conv2D(
            32, 3, padding="same", use_bias=False
        ),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Activation("relu"),
        tf.keras.layers.MaxPooling2D(2),

        tf.keras.layers.Conv2D(
            64, 3, padding="same", use_bias=False
        ),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Activation("relu"),
        tf.keras.layers.MaxPooling2D(2),

        tf.keras.layers.Conv2D(
            128, 3, padding="same", use_bias=False
        ),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Activation("relu"),
        tf.keras.layers.MaxPooling2D(2),

        tf.keras.layers.Conv2D(
            192, 3, padding="same", use_bias=False
        ),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Activation("relu"),
        tf.keras.layers.MaxPooling2D(2),

        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.Dense(10, activation="softmax")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model


def make_fold_dataset(frame, training=False):
    """Create a fresh dataset using explicit paths and labels."""
    paths = frame["path"].to_numpy()
    labels = frame["gesture"].map(class_to_idx).to_numpy(
        dtype=np.int32
    )

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    if training:
        ds = ds.shuffle(
            buffer_size=len(frame),
            seed=42,
            reshuffle_each_iteration=True
        )

    def load_image(path, label):
        image = tf.io.read_file(path)
        image = tf.io.decode_png(image, channels=1)
        image = tf.image.resize(
            image, [IMG_HEIGHT, IMG_WIDTH]
        )
        image = tf.cast(image, tf.float32) / 255.0
        return image, label

    ds = ds.map(
        load_image,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


# Inspect fold 1 without training
fold = folds[0]

fit_df_fold = df[
    df["subject"].isin(fold["fit_subjects"])
].copy()

val_df_fold = df[
    df["subject"] == fold["validation_subject"]
].copy()

test_df_fold = df[
    df["subject"] == fold["test_subject"]
].copy()

assert set(fit_df_fold["subject"]).isdisjoint(
    val_df_fold["subject"]
)
assert set(fit_df_fold["subject"]).isdisjoint(
    test_df_fold["subject"]
)
assert set(val_df_fold["subject"]).isdisjoint(
    test_df_fold["subject"]
)

train_ds_fold = make_fold_dataset(
    fit_df_fold, training=True
)
val_ds_fold = make_fold_dataset(val_df_fold)
test_ds_fold = make_fold_dataset(test_df_fold)

images, labels = next(iter(train_ds_fold))
model_check = build_cnn()

print("Fold configuration:")
print("Fitting subjects:", fold["fit_subjects"])
print("Validation subject:", fold["validation_subject"])
print("Test subject:", fold["test_subject"])

print("\nImage counts:")
print("Fit:", len(fit_df_fold))
print("Validation:", len(val_df_fold))
print("Test:", len(test_df_fold))

print("\nTraining batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Model input shape:", model_check.input_shape)
print("Model output shape:", model_check.output_shape)
print("Model parameters:", model_check.count_params())

assert images.shape == (BATCH_SIZE, 96, 256, 1)
assert model_check.output_shape == (None, 10)

print("\nFold pipeline verification passed.")


In [ ]:

from collections import defaultdict
import pandas as pd

# Verify that the previous exact-hash audit is available
assert "hash_to_paths" in globals(), (
    "Please rerun the original exact-duplicate hash audit first."
)

path_to_hash = {
    path: digest
    for digest, paths in hash_to_paths.items()
    for path in paths
}

subject_by_path = dict(zip(df["path"], df["subject"]))

results = []

for i, fold in enumerate(folds, start=1):
    role_by_subject = {
        s: "fit" for s in fold["fit_subjects"]
    }
    role_by_subject[fold["validation_subject"]] = "validation"
    role_by_subject[fold["test_subject"]] = "test"

    roles_by_hash = defaultdict(set)

    for path, digest in path_to_hash.items():
        role = role_by_subject[subject_by_path[path]]
        roles_by_hash[digest].add(role)

    crossing = [
        roles for roles in roles_by_hash.values()
        if len(roles) > 1
    ]

    results.append({
        "fold": i,
        "test_subject": fold["test_subject"],
        "cross_split_duplicate_groups": len(crossing),
        "fit_test": sum(
            {"fit", "test"}.issubset(r) for r in crossing
        ),
        "validation_test": sum(
            {"validation", "test"}.issubset(r) for r in crossing
        ),
        "fit_validation": sum(
            {"fit", "validation"}.issubset(r) for r in crossing
        )
    })

audit_df = pd.DataFrame(results)
display(audit_df)

print("\nTotal cross-split duplicate groups across folds:",
      audit_df["cross_split_duplicate_groups"].sum())
print("Folds with fitting-test duplicates:",
      (audit_df["fit_test"] > 0).sum())
print("Folds with validation-test duplicates:",
      (audit_df["validation_test"] > 0).sum())


In [ ]:

from collections import defaultdict
import pandas as pd

# Map each exact image hash to its subjects and paths
hash_to_records = defaultdict(list)

for row in df[["path", "subject", "gesture"]].itertuples(index=False):
    path, subject, gesture = row
    digest = path_to_hash[path]

    hash_to_records[digest].append({
        "path": path,
        "subject": subject,
        "gesture": gesture
    })

duplicate_details = []

for digest, records in hash_to_records.items():
    subjects_in_group = sorted({
        record["subject"] for record in records
    })

    if len(subjects_in_group) > 1:
        duplicate_details.append({
            "hash": digest,
            "subjects": subjects_in_group,
            "gestures": sorted({
                record["gesture"] for record in records
            }),
            "paths": [record["path"] for record in records]
        })

print("Exact duplicate groups across subjects:",
      len(duplicate_details))

for item in duplicate_details[:10]:
    print("\nSubjects:", item["subjects"])
    print("Gestures:", item["gestures"])
    for path in item["paths"]:
        print(" ", path)


In [ ]:

# Count duplicate image files belonging to subjects 02 and 05

affected_subjects = {"02", "05"}

affected_df = df[
    df["subject"].isin(affected_subjects)
].copy()

duplicate_hashes = {
    digest
    for digest, records in hash_to_records.items()
    if len({r["subject"] for r in records}) > 1
}

affected_duplicate_records = []

for digest in duplicate_hashes:
    for record in hash_to_records[digest]:
        if record["subject"] in affected_subjects:
            affected_duplicate_records.append({
                "hash": digest,
                "subject": record["subject"],
                "gesture": record["gesture"],
                "path": record["path"]
            })

affected_duplicates_df = pd.DataFrame(
    affected_duplicate_records
)

print("Images belonging to subjects 02 and 05:",
      len(affected_df))

print("Cross-subject duplicate hash groups:",
      len(duplicate_hashes))

print("\nDuplicate files by subject and gesture:")
print(
    affected_duplicates_df
    .groupby(["subject", "gesture"])
    .size()
)

print("\nUnique duplicate hashes by gesture:")
print(
    affected_duplicates_df
    .drop_duplicates("hash")
    .groupby("gesture")
    .size()
)


In [ ]:

print("Folds affected by the subject 02/05 duplicate issue:\n")

for i, fold in enumerate(folds, start=1):
    fit_subjects = set(fold["fit_subjects"])
    test_subject = fold["test_subject"]
    val_subject = fold["validation_subject"]

    affected = (
        ("02" in fit_subjects and test_subject == "05")
        or
        ("05" in fit_subjects and test_subject == "02")
    )

    if affected:
        print({
            "fold": i,
            "fit_contains_02": "02" in fit_subjects,
            "fit_contains_05": "05" in fit_subjects,
            "validation_subject": val_subject,
            "test_subject": test_subject
        })


In [ ]:

import os
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import accuracy_score, f1_score

# Explicit settings
EPOCHS = 30
PATIENCE = 5
OUTPUT_DIR = "/kaggle/working/loso_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Reuse the verified fold and dataset functions
required = [
    "folds", "df", "classes", "class_to_idx",
    "make_fold_dataset", "build_cnn"
]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(f"Missing required variables: {missing}")

results_path = os.path.join(OUTPUT_DIR, "fold_results.csv")

# Resume completed folds if the notebook is interrupted
if os.path.exists(results_path):
    results_df = pd.read_csv(results_path)
    completed = set(results_df["fold"].astype(int))
else:
    results_df = pd.DataFrame()
    completed = set()

for fold_number, fold in enumerate(folds, start=1):
    if fold_number in completed:
        print(f"Skipping completed fold {fold_number}/10")
        continue

    print(f"\n{'='*55}")
    print(f"Fold {fold_number}/10")
    print("Validation subject:", fold["validation_subject"])
    print("Test subject:", fold["test_subject"])

    fit_df_fold = df[
        df["subject"].isin(fold["fit_subjects"])
    ].copy()
    val_df_fold = df[
        df["subject"] == fold["validation_subject"]
    ].copy()
    test_df_fold = df[
        df["subject"] == fold["test_subject"]
    ].copy()

    # Assert strict subject separation
    assert set(fit_df_fold["subject"]).isdisjoint(
        val_df_fold["subject"]
    )
    assert set(fit_df_fold["subject"]).isdisjoint(
        test_df_fold["subject"]
    )
    assert set(val_df_fold["subject"]).isdisjoint(
        test_df_fold["subject"]
    )

    train_ds_fold = make_fold_dataset(
        fit_df_fold, training=True
    )
    val_ds_fold = make_fold_dataset(val_df_fold)
    test_ds_fold = make_fold_dataset(test_df_fold)

    tf.keras.backend.clear_session()
    model_fold = build_cnn()

    checkpoint_path = os.path.join(
        OUTPUT_DIR, f"best_fold_{fold_number}.keras"
    )

    callbacks_fold = [
        tf.keras.callbacks.ModelCheckpoint(
            checkpoint_path,
            monitor="val_loss",
            save_best_only=True,
            verbose=0
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=PATIENCE,
            restore_best_weights=True,
            verbose=1
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            min_lr=1e-6,
            verbose=0
        )
    ]

    history_fold = model_fold.fit(
        train_ds_fold,
        validation_data=val_ds_fold,
        epochs=EPOCHS,
        callbacks=callbacks_fold,
        verbose=1
    )

    # Evaluate the best checkpoint on the held-out subject
    best_fold_model = tf.keras.models.load_model(
        checkpoint_path
    )

    y_true = np.concatenate([
        y.numpy() for _, y in test_ds_fold
    ])
    probabilities = best_fold_model.predict(
        test_ds_fold, verbose=0
    )
    y_pred = np.argmax(probabilities, axis=1)

    fold_accuracy = accuracy_score(y_true, y_pred)
    fold_macro_f1 = f1_score(
        y_true, y_pred, average="macro"
    )

    result = {
        "fold": fold_number,
        "test_subject": fold["test_subject"],
        "validation_subject": fold["validation_subject"],
        "accuracy": fold_accuracy,
        "macro_f1": fold_macro_f1,
        "epochs_completed": len(history_fold.history["loss"]),
        "best_validation_accuracy": max(
            history_fold.history["val_accuracy"]
        ),
        "duplicate_affected": fold["test_subject"] in {"02", "05"}
    }

    results_df = pd.concat(
        [results_df, pd.DataFrame([result])],
        ignore_index=True
    )
    results_df.to_csv(results_path, index=False)

    print(
        f"Fold {fold_number} accuracy: "
        f"{fold_accuracy * 100:.2f}%"
    )
    print(f"Fold {fold_number} macro F1: {fold_macro_f1:.4f}")
    print("Saved results:", results_path)

    del model_fold, best_fold_model
    del train_ds_fold, val_ds_fold, test_ds_fold
    tf.keras.backend.clear_session()

print("\nAll folds processed.")
display(
    results_df.sort_values("fold").reset_index(drop=True)
)
